# ELAsTiCC2 — Exploration and Mini-Classification

- **Author**: Sylvie Dagoret-Campagne
- **Affiliation**: IJCLab/IN2P3/CNRS
- **Created**: 2025-11-06 at NERSC (kernel `desc-td-env-dev`)
- **Updated**: 2025-11-10 on laptop (kernel `pytorch-cpu-py312`)
- **Updated**: 2025-12-31 at CCIN2P3 (kernel `desc_2026_py312`)
- **Updated**: 2026-03-30 — refactored for portability between NERSC and CCIN2P3

## Overview

This notebook demonstrates how to:
1. Detect the current computing environment (NERSC or CCIN2P3) and set data paths accordingly.
2. Explore the ELAsTiCC2 training sample (FITS format) — listing available supernova types,
   reading photometric light curves and associated header metadata.
3. Visualise multi-band light curves.
4. Extract simple per-band features (peak flux, peak MJD) and global object properties
   (number of observations, redshift).
5. Train a Random Forest classifier on those features and evaluate its performance.

The notebook is designed to run **transparently on NERSC, CCIN2P3, or a local laptop**
without any manual path editing.

- https://portal.nersc.gov/cfs/lsst/DESC_TD_PUBLIC/ELASTICC/
- https://portal.nersc.gov/cfs/lsst/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2/
- https://github.com/LSSTDESC/elasticc
- https://github.com/LSSTDESC/elasticc/tree/lib_elasticc2
- https://github.com/LSSTDESC/elasticc/blob/main/taxonomy/taxonomy.ipynb
- https://github.com/LSSTDESC/elasticc_metrics

## 1. Imports

In [ ]:
# ============================================================
# Standard library and third-party imports
# ============================================================
import os
import sys
import socket
import random

import numpy as np
import pandas as pd
import fitsio
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility seeds
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Python executable : {sys.executable}")
print(f"NumPy version     : {np.__version__}")
print(f"Pandas version    : {pd.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

## 2. Environment Detection and Data Paths

The helper functions below inspect the hostname and environment variables to identify
whether we are running at **NERSC** (Perlmutter/Cori), at **CCIN2P3**, or on a
**local laptop/workstation**.  
The `BASE_PATH` variable is then set to the appropriate location of the
`ELASTICC2_TRAINING_SAMPLE_2` directory on each system.

In [ ]:
# ============================================================
# Environment detection helpers
# ============================================================

def is_on_nersc() -> bool:
    """Return True when running on a NERSC compute node (Perlmutter or Cori)."""
    hostname = socket.gethostname()
    # Check the hostname against known NERSC patterns
    nersc_hostname_patterns = ["perlmutter", "cori", "nersc.gov"]
    host_match = any(pattern in hostname for pattern in nersc_hostname_patterns)
    # Also check for environment variables set by NERSC batch systems
    nersc_env_vars = ["NERSC_HOST", "SLURM_JOB_ID", "CRAY_SYSTEM_NAME"]
    env_match = any(var in os.environ for var in nersc_env_vars)
    return host_match or env_match


def is_on_ccin2p3() -> bool:
    """Return True when running on a CCIN2P3 batch/interactive node."""
    hostname = socket.gethostname()
    # CCIN2P3 nodes have hostnames ending in 'cc.in2p3.fr'
    # Additionally, PBS-based jobs set PBS_* variables; we check for PBS_O_WORKDIR
    host_match = "cc.in2p3.fr" in hostname
    env_match  = "PBS_O_WORKDIR" in os.environ  # set by the PBS job scheduler
    return host_match or env_match


# ============================================================
# Select the correct root path for ELASTICC2 data
# ============================================================

if is_on_nersc():
    SITE = "NERSC"
    BASE_PATH = "/global/cfs/cdirs/lsst/www/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
elif is_on_ccin2p3():
    SITE = "CCIN2P3"
    BASE_PATH = "/sps/lsst/groups/desc/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
else:
    SITE = "Laptop / other"
    # Adjust this path if you have a local copy of the data
    BASE_PATH = "/Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"

print(f"Running on  : {SITE}")
print(f"Hostname    : {socket.gethostname()}")
print(f"BASE_PATH   : {BASE_PATH}")
print(f"Path exists : {os.path.isdir(BASE_PATH)}")

## 3. Available Supernova Types

The training sample is organised as one sub-directory per transient class
(e.g. `ELASTICC2_TRAIN_02_SNIa-SALT3`, `ELASTICC2_TRAIN_02_SNIc-Templates`, …).
Each sub-directory contains pairs of compressed FITS files:
- `*_HEAD.FITS.gz` — object-level header table (one row per object)
- `*_PHOT.FITS.gz` — photometric observations table (all bands, all epochs)

In [ ]:
# List all transient-class sub-directories present under BASE_PATH
all_sn_types = sorted(
    d for d in os.listdir(BASE_PATH)
    if os.path.isdir(os.path.join(BASE_PATH, d))
)

print(f"Total number of transient classes found : {len(all_sn_types)}")
print("First 10 classes:")
for t in all_sn_types[:10]:
    print(f"  {t}")

## 4. Select a Subset of Classes for This Notebook

For demonstration purposes we work with three classes only:
- Type Ia SNe modelled with SALT3
- Type Ic SNe using template light curves
- Type Ib SNe using template light curves

Adjust `SAMPLE_TYPES` to include more (or different) classes.

In [ ]:
# Subset of transient classes used in this notebook
SAMPLE_TYPES = [
    "ELASTICC2_TRAIN_02_SNIa-SALT3",
    "ELASTICC2_TRAIN_02_SNIc-Templates",
    "ELASTICC2_TRAIN_02_SNIb-Templates",
]

# Verify that each requested class actually exists on disk
for sn_type in SAMPLE_TYPES:
    path = os.path.join(BASE_PATH, sn_type)
    status = "OK" if os.path.isdir(path) else "MISSING"
    print(f"  [{status}]  {sn_type}")

## 5. Inspect the FITS File Structure

We open the first `HEAD` and `PHOT` files for the SNIa class and print
their column names, so we know which fields are available downstream.
We also retrieve the set of photometric bands present in the PHOT table.

In [ ]:
# Use the SNIa class as a representative example
EXAMPLE_TYPE = "ELASTICC2_TRAIN_02_SNIa-SALT3"
example_dir  = os.path.join(BASE_PATH, EXAMPLE_TYPE)

# Collect HEAD and PHOT file lists (sorted to keep HEAD/PHOT pairs aligned)
head_files = sorted(f for f in os.listdir(example_dir) if "HEAD" in f)
phot_files = sorted(f for f in os.listdir(example_dir) if "PHOT" in f)

print(f"Number of HEAD files in {EXAMPLE_TYPE}: {len(head_files)}")
print(f"Number of PHOT files in {EXAMPLE_TYPE}: {len(phot_files)}")

# Read the first pair
head_data = fitsio.FITS(os.path.join(example_dir, head_files[0]))[1].read()
phot_data = fitsio.FITS(os.path.join(example_dir, phot_files[0]))[1].read()

print("\nHEAD columns:")
print(head_data.dtype.names)

print("\nPHOT columns:")
print(phot_data.dtype.names)

# Retrieve unique (non-dash) photometric bands, stripping whitespace
bands_available = sorted(
    {b.strip() for b in phot_data["BAND"] if b.strip() not in ("", "-")}
)
print(f"\nPhotometric bands present: {bands_available}")

In [ ]:
len(head_data)

In [ ]:
phot_data

## 6. Visualise an Example Multi-Band Light Curve

We plot `FLUXCAL` vs `MJD` for each photometric band, with error bars from `FLUXCALERR`.

In [ ]:
print("SNID unique dans les données HEAD:", len(np.unique(head_data['SNID']))," :: " ,np.unique(head_data['SNID']))
#print("SNID unique dans les données PHOT:", np.unique(phot_data['SNID']))

In [ ]:
phot_data["BAND"]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for band in bands_available:
    # Select rows corresponding to this band (strip trailing spaces)
    mask = np.array([b.strip() for b in phot_data["BAND"]]) == band
    ax.errorbar(
        phot_data["MJD"][mask],
        phot_data["FLUXCAL"][mask],
        yerr=phot_data["FLUXCALERR"][mask],
        fmt="o",
        label=f"Band {band}",
        capsize=3,
    )

ax.grid()
ax.set_xlabel("MJD (Modified Julian Date)")
ax.set_ylabel("FLUXCAL (calibrated flux)")
ax.set_title(f"Example multi-band light curve — {EXAMPLE_TYPE}")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
assert False

## 7. Feature Extraction

For each object we extract the following simple, band-agnostic features:
- **`flux_max_{band}`** — maximum calibrated flux observed in that band.
- **`mjd_peak_{band}`** — MJD at which the maximum flux is reached.
- **`NOBS`** — total number of observations (from the header table).
- **`REDSHIFT_FINAL`** — best-estimate redshift (from the header table).

These features are stored in a `pandas` DataFrame `X` with a corresponding label
array `y` (the transient class name).

In [ ]:
def extract_features_from_file_pair(
    head_path: str,
    phot_path: str,
    bands: list,
    label: str,
) -> tuple:
    """Read one HEAD/PHOT pair and return (list_of_feature_dicts, list_of_labels)."""
    head_data = fitsio.FITS(head_path)[1].read()
    phot_data = fitsio.FITS(phot_path)[1].read()

    # Pre-compute stripped band column once for efficiency
    phot_bands_stripped = np.array([b.strip() for b in phot_data["BAND"]])

    features_list = []
    labels_list   = []

    for row in head_data:
        feature_dict = {}

        # Per-band peak flux and MJD of peak
        for band in bands:
            mask = phot_bands_stripped == band
            if mask.sum() > 0:
                flux   = phot_data["FLUXCAL"][mask]
                mjd    = phot_data["MJD"][mask]
                peak_i = int(np.argmax(flux))
                feature_dict[f"flux_max_{band}"] = float(flux[peak_i])
                feature_dict[f"mjd_peak_{band}"] = float(mjd[peak_i])
            else:
                # Band not observed in this file — fill with zeros
                feature_dict[f"flux_max_{band}"] = 0.0
                feature_dict[f"mjd_peak_{band}"] = 0.0

        # Global object properties from the header
        feature_dict["NOBS"]           = int(row["NOBS"])
        feature_dict["REDSHIFT_FINAL"] = float(row["REDSHIFT_FINAL"])

        features_list.append(feature_dict)
        labels_list.append(label)

    return features_list, labels_list


# ── Build the full feature matrix over all selected classes ──
all_features = []
all_labels   = []

for sn_type in SAMPLE_TYPES:
    type_dir   = os.path.join(BASE_PATH, sn_type)
    head_files = sorted(f for f in os.listdir(type_dir) if "HEAD" in f)
    phot_files = sorted(f for f in os.listdir(type_dir) if "PHOT" in f)

    print(f"Processing {sn_type} ({len(head_files)} file pair(s)) ...", end=" ")

    for head_file, phot_file in zip(head_files, phot_files):
        feats, labs = extract_features_from_file_pair(
            head_path=os.path.join(type_dir, head_file),
            phot_path=os.path.join(type_dir, phot_file),
            bands=bands_available,
            label=sn_type,
        )
        all_features.extend(feats)
        all_labels.extend(labs)

    print("done.")

# Assemble into a DataFrame
X = pd.DataFrame(all_features)
y = np.array(all_labels)

print(f"\nFeature matrix shape : {X.shape}")
print("Class distribution:")
for cls, cnt in zip(*np.unique(y, return_counts=True)):
    print(f"  {cls}: {cnt} objects")
print("\nFirst 5 rows:")
X.head()

## 8. Train / Test Split

We split the dataset 70 % / 30 % using stratified sampling so that each class
is proportionally represented in both the training and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y,
)

print(f"Training set size : {len(X_train)} objects")
print(f"Test set size     : {len(X_test)} objects")

## 9. Random Forest Classifier

We train a simple Random Forest with 100 estimators.  Despite its simplicity
this approach provides a useful baseline for comparing more sophisticated
time-series or deep-learning classifiers.

In [ ]:
# Instantiate and train the classifier
clf = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_SEED,
    n_jobs=-1,   # use all available CPU cores
)
clf.fit(X_train, y_train)

print("Training complete.")
print(f"Number of features used : {clf.n_features_in_}")
print(f"Number of estimators    : {clf.n_estimators}")

## 10. Evaluation on the Test Set

We report the standard per-class precision, recall and F1-score, as well as
the confusion matrix.  Short class labels are used for readability.

In [ ]:
# Predict on the held-out test set
y_pred = clf.predict(X_test)

# Shorten class names for display
short_names = [name.replace("ELASTICC2_TRAIN_02_", "") for name in np.unique(y)]

print("=" * 60)
print("Classification report")
print("=" * 60)
print(
    classification_report(
        y_test,
        y_pred,
        target_names=short_names,
    )
)

print("=" * 60)
print("Confusion matrix (rows = true class, cols = predicted class)")
print("=" * 60)
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=short_names, columns=short_names)
print(cm_df)

## 11. Feature Importances

The Random Forest can rank features by their contribution to the splitting
decisions (Mean Decrease in Impurity).  The top-10 most discriminative
features are displayed below.

In [ ]:
importances = pd.Series(
    clf.feature_importances_,
    index=X.columns,
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
importances.head(10).plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Top-10 Feature Importances (Mean Decrease in Impurity)")
ax.set_ylabel("Importance")
ax.set_xlabel("Feature")
plt.tight_layout()
plt.show()

print("\nTop-10 features:")
print(importances.head(10).to_string())